## Beaty Informatics x MDS Capstone: Proposal

### Executive Summary
##### A brief overview of the problem, your proposed approach, and the expected outcome

Taxonomy, including species names and ranks, is constantly changing with the advent of new scientific methods. Curators at the Beaty Biodiversity Museum have to keep track of all of these changes in order to keep the collections current, but finding and evaluating information can be time consuming. We propose creating a web app which synthesizes current information online regarding species name synonyms and taxonomy and provides it to curators, who can then decide which designation they'd like to incorporate into the museum's database. 


### Introduction
##### Describe the problem in your own words, Explain why it is important, Provide relevant background
- Taxonomy refers to naming, describing, and classifying organisms into hierarchical groups based on shared characteristics and evolutionary relationships.
- Museum curators are responsible for keeping organized, up-to-date, searchable collections for researchers and experts to use, with  up-to-date taxonomic classifications and names. 
- Individual species names can change over time, introducing synonyms, basionyms, variants, and subspecies which can get in the way of curator's overall goals. 
- taxonomic classifications (e.g. family, order) can also change, introducing further uncertainty into where organisms should physically and digitally be stored. 
- Databases such as GBIF (Global Biodiversity Information Facility) and GenBank, as well as more taxa-specific sites such as Mushroom Observer and BugGuide, publish updated species names, name changes, and taxonomic classification changes

Refine the problem into specific objectives
- When a specimen comes into the museums, curators are responsible for identifying it and classifying it correctly. When they are unsure if the name or classification has changed, that's where we step in. By making multiple API calls to different sites, both taxa specific and not, we can provide a list of species synonyms which are online. 
- Curators can select which sites they prefer to refer to, and see species synonyms.
- Synonyms are the first step of our project; we hope to expand into species taxonomy and even ranges. 

Describe the final data product (e.g., dashboard, model, pipeline)
- The final data product is a web app which synthesizes online information with a search. 

### Data & Exploratory Data Analysis (EDA)
Provide a clear description and initial exploration of your data:
- Data source(s), size, and structure

Data sources vary by taxonomic group, with some sources in common between them (for example, GBIF, GenBank). API calls return JSON-formatted species synonyms, variants, and subspecies. The output is structured as synonyms being the key, and variants / subspecies are the values. Size varies by species - names could have 0 variants, or dozens. Certain species may have no synonyms, some have plenty. 

- Definition of the observational unit

An observation unit, in this case, could be considered either species, or their name synonyms. 

- Key variables/features


Include tables or visualizations showing data quality issues (e.g., missing values, imbalance) and patterns or signals relevant to your task. Also, discuss limitations of the data and potential challenges and their implications. Avoid including raw code in the report. Instead, use a pipeline to generate figures and results.

In [1]:
import requests
import pandas as pd
import sys
sys.path.insert(0, "..")

from scripts.APIs.GBIF import get_gbif_synonyms

In [4]:
# we should probably change this to be not JUST the gbif function. 
result = get_gbif_synonyms("Amanita muscaria")

def display_synonyms_markdown(synonyms: dict, query: str):
    lines = [f"## Synonyms for *{query}*\n"]
    for species, variants in synonyms.items():
        lines.append(f"- **{species}**")
        for v in variants:
            lines.append(f"  - *{v}*")
    from IPython.display import Markdown, display
    display(Markdown("\n".join(lines)))

display_synonyms_markdown(result, "Amanita muscaria")

## Synonyms for *Amanita muscaria*

- **Agaricus aureolus**
- **Agaricus imperialis**
- **Agaricus muscarius**
  - *Agaricus muscarius formosus*
  - *Agaricus muscarius puella*
  - *Agaricus muscarius sanguineus*
- **Agaricus nobilis**
- **Agaricus pseudoaurantiacus**
- **Agaricus puellus**
- **Amanita aureola**
- **Amanita circinnata**
- **Amanita formosa**
- **Amanita muscaria**
  - *Amanita muscaria aureola*
  - *Amanita muscaria beglyanovae*
  - *Amanita muscaria eu-umbrina*
  - *Amanita muscaria europaea*
  - *Amanita muscaria flavivolvata*
  - *Amanita muscaria formosa*
  - *Amanita muscaria guessowii*
  - *Amanita muscaria puella*
  - *Amanita muscaria vaginata*
  - *Amanita muscaria americana*
  - *Amanita muscaria hercynica*
  - *Amanita muscaria minor*
  - *Amanita muscaria sudedica*
  - *Amanita muscaria umbrina*
  - *Amanita muscaria alba*
  - *Amanita muscaria alpha*
  - *Amanita muscaria coccinea*
  - *Amanita muscaria fuligineoverrucosa*
  - *Amanita muscaria sanguinea*
  - *Amanita muscaria speciosa*
  - *Amanita muscaria tomentosa*
  - *Amanita muscaria vulgaris*
- **Amanita puella**
- **Amanitaria muscaria**
- **Hypophyllum muscarium**
- **Venenarius muscarius**

In [5]:
df = pd.DataFrame([
    {"Species": species, "# Variants": len(variants), "Variants": ", ".join(variants) or "—"}
    for species, variants in result.items()
])
df

,Species,# Variants,Variants
0,Agaricus aureolus,0,—
1,Agaricus imperialis,0,—
2,Agaricus muscarius,3,"Agaricus muscarius formosus, Agaricus muscariu..."
3,Agaricus nobilis,0,—
4,Agaricus pseudoaurantiacus,0,—
5,Agaricus puellus,0,—
6,Amanita aureola,0,—
7,Amanita circinnata,0,—
8,Amanita formosa,0,—
9,Amanita muscaria,22,"Amanita muscaria aureola, Amanita muscaria beg..."


### Data Science Approach

The core need we identified through conversations with curators at the Beaty Biodiversity Museum is the ability to reconcile their internal specimen records against multiple external data authorities. When a curator is reviewing a species determination (for example, deciding whether a mushroom specimen should be classified as Amanita muscaria or under a different name used by another source), they currently have to manually check several outside sources. Our goal is to build a data pipeline that automates the collection and comparison of records from these external sources, so that curators can make better-informed, better-documented decisions about what is correct for the Beaty's database.
 
Our approach begins with understanding which external databases are relevant to each area of study. Biological collections span a wide range of taxa (fungi, plants, birds, marine life, and more), and each field has its own set of authoritative sources. In the first week of the project, our team explored four databases (GBIF, GenBank, Mushroom Observer, and iNaturalist) and tested their APIs using the species Amanita muscaria as a test case. The API responses return JSON data which we were able to process and compare across sources. As the project continues, we will expand this to other databases to build a more complete list of the sources. Each source will have its own script responsible for retrieving relevant records and returning them in a consistent format for downstream processing.
 
Once data is collected from each source, it needs to be normalized and cross-referenced so that meaningful comparisons can be made against Beaty's own records. This is non-trivial as each database has its own schema. Our pipeline will need to handle these differences carefully, matching records across sources using identifiers such as species names and collection metadata. As our codebase grows, we may have to define a Python abstract class that each database-specific subclass must implement. This ensures that as new databases are added to the pipeline in the future, they conform to the same contract: for example, every database subclass will be expected to retrieve synonyms, return normalized records, and expose provenance metadata. This makes the codebase easier to extend and maintain over time.
 
A key part of what we are building is not just data retrieval, but provenance modelling: tracking where each piece of data came from, when it was recorded, and by whom. The capstone partner has emphasized that authorities change over time, as data sources become stale, funding lapses, and scientific consensus shifts. Our pipeline will attach provenance information to each reconciled record so that a curator reviewing a proposed change can see, for example, that a new determination came from a record on Mushroom Observer in November 2025, and can weigh that against Beaty's own record from 2019. This provenance layer is what separates our tool from a simple data aggregator, as it gives the curator the context they need to make a defensible decision rather than just accepting data at face value.
 
To present the reconciled information to curators, our team has been exploring several interface designs: a table showing how a species name appears across all sources, a node-based expandable graph showing results from different sources, and a summarized view powered by a Retrieval-Augmented Generation (RAG) approach where an LLM synthesizes the returned data into a plain-language summary. In all designs, our priority is to surface information without introducing bias. Aggregated summaries in particular carry a risk of obscuring disagreement between sources, so we will be thoughtful about how we present confidence and provenance alongside any summary. Ultimately, the curator remains the decision-maker, as our tool is meant to give them better information, not to make determinations on their behalf.

### Evaluation & Success Criteria

We define success in terms of the reliability and completeness of the information we surface, and whether that information genuinely helps curators make better and more informed decisions. Our primary success condition is that the pipeline can accurately retrieve, normalize, and compare records from the configured data sources, and present them to curators in a way that is clear, transparent about provenance, and free of systematic bias.
 
To support curator decision-making, each candidate name or record returned by the pipeline may be accompanied by metrics that reflect its standing across sources. For example, we may try two main metrics for each result: coverage (how many databases list this name, which reflects how widely recognized it is across the scientific community) and recency (how recently the name was published or updated, where that information is available). Together, these metrics give curators a sense of which names are broadly recognized and current, without making an opinionated claim about which answer is "right." The curator can use these metrics to quickly identify which names are well-supported.
 
One way we will track our own progress throughout the project is by maintaining a table of all databases we intend to integrate, with each row tracking both connection status (does the pipeline successfully pull data from this source?) and functional completeness (does it expose all the required methods defined in the abstract base class?). This gives us a clear, measurable picture of how much of the planned scope we have covered at any given point. For databases that do not expose a public API, we will explore alternative approaches such as SQL exports or custom retrieval scripts, and document these cases explicitly so that future maintainers understand why they differ from the standard pattern.
 
We will also evaluate success through feedback from the curators at the Beaty. Our capstone partner plans to share the pipeline with curators working across different collections (fungi, plants, marine specimens). Their ability to use the tool effectively, and their confidence in the provenance and authority information it surfaces, is our most important qualitative measure of success. If a curator can look at the output of the pipeline, understand where each piece of data came from, and feel that the tool has improved their ability to make an informed determination, then we have met the spirit of what the project is asking for.
 
The final deliverable will be a working pipeline and app hosted on Beaty's GitHub, with documentation that allows curators and future developers to use and extend it. A curator should be able to enter a species name and retrieve a reconciled view of relevant records from all configured sources, along with provenance metadata for each result.

### Timeline
Provide a high-level timeline outlining key milestones, expected deliverables, and the overall sequence of work.
